In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print("====== STEP 1: PREPARING BENCHMARK AND OFFICIAL PROMPTAD ======")
BENCHMARK_REPOSITORY = "Parsagh05/Natural-Corruption-Robustness"
BENCHMARK_ROOT = Path("/kaggle/working/Natural-Corruption-Robustness")
PROMPTAD_ROOT = Path("/kaggle/working/PromptAD")
PROMPTAD_COMMIT = "0f86ce0dc1ed59007d51348d8d566aed31360cf9"

if not BENCHMARK_ROOT.exists():
    subprocess.run(["git", "clone", f"https://github.com/{BENCHMARK_REPOSITORY}.git", str(BENCHMARK_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(BENCHMARK_ROOT), "pull", "--ff-only"], check=True)
required_harness_file = BENCHMARK_ROOT / "few_shot" / "harness" / "models.py"
required_runner_file = BENCHMARK_ROOT / "few_shot" / "harness" / "runner.py"
for required_file in (required_harness_file, required_runner_file):
    if not required_file.is_file():
        raise RuntimeError(
            "The cloned benchmark does not contain the PromptAD few-shot implementation. "
            "Commit and push the local few_shot changes before running Kaggle. "
            f"Missing: {required_file}"
        )
if "discover_promptad_checkpoints" not in required_harness_file.read_text(encoding="utf-8"):
    raise RuntimeError("The cloned benchmark revision predates the PromptAD evaluation wrapper.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
     "-r", str(BENCHMARK_ROOT / "few_shot" / "requirements.txt")],
    check=True,
)
# Match the versions recorded by the checkpoint training manifests.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
     "open_clip_torch==2.32.0", "timm==1.0.26"],
    check=True,
)

if not PROMPTAD_ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/FuNz-0/PromptAD.git", str(PROMPTAD_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROMPTAD_ROOT), "fetch", "origin"], check=True)
subprocess.run(["git", "-C", str(PROMPTAD_ROOT), "checkout", "--detach", PROMPTAD_COMMIT], check=True)
resolved_commit = subprocess.run(
    ["git", "-C", str(PROMPTAD_ROOT), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if resolved_commit != PROMPTAD_COMMIT:
    raise RuntimeError(f"Wrong PromptAD source commit: {resolved_commit}")

for import_path in (BENCHMARK_ROOT, PROMPTAD_ROOT):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))
os.environ["PROMPTAD_ROOT"] = str(PROMPTAD_ROOT)

import numpy as np
import torch
from PIL import Image
from shared.corruption import apply_corruption

if not torch.cuda.is_available():
    raise RuntimeError(
        "PromptAD's official implementation is fp16-only. In Kaggle, "
        "open Settings -> Accelerator and select a GPU."
    )
smoke_pixels = np.random.default_rng(0).integers(0, 256, (64, 96, 3), dtype=np.uint8)
smoke_image = Image.fromarray(smoke_pixels)
smoke_operations = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast", "rotation", "zooming", "shifting",
]
for operation in smoke_operations:
    smoke_result = apply_corruption(smoke_image, operation, 1, "smoke.png", 123)
    if smoke_result.size != smoke_image.size:
        raise RuntimeError(
            f"Corruption smoke test changed dimensions for {operation}: "
            f"{smoke_image.size} -> {smoke_result.size}"
        )
print(f"Corruption smoke test passed for {len(smoke_operations)} operations.")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Official PromptAD commit: {resolved_commit}")
print("Environment ready.")


## Fixed evaluation protocol

Attach the Kaggle dataset **PromptAD Few-Shot Checkpoints — MVTec AD & VisA** (`parsagholami/promptad-few-shot-checkpoints-mvtec-ad-and-visa`), plus the selected MVTec AD and/or VisA image datasets. Enable Internet because the notebook pins the official PromptAD source and downloads the frozen `ViT-B-16-plus-240 / laion400m_e32` backbone, which is intentionally not duplicated inside the small checkpoint files.

The control block below mirrors `kaggle_inpformer.ipynb`: dataset, clean baseline, categorized or uncategorized corruptions, subsets, severities, shots, device, batch size, cache, and hash verification are editable. PromptAD uses paired class-specific checkpoints: `CLS` supplies the harmonic fusion of its textual score and maximum visual-map score, while `SEG` supplies the final 400 × 400 pixel map. Artifacts retain the native 15 × 15 fused SEG map and apply the official interpolation and smoothing only for metrics. The wrapper retains the released test-code preprocessing and scoring path, including its cv2 1024-square pre-resize, BGR channel behavior, Gaussian sigma 4 segmentation smoothing, and separate task buffers. A complete both-dataset, three-shot corruption run is large; use one value in `SHOTS_TO_RUN` per Kaggle session when necessary.


In [ ]:
import gc

from few_shot.harness.dataset import AnomalyDetectionDataset, build_dataset_configs
from few_shot.harness.models import discover_promptad_checkpoints
from few_shot.harness.runner import run_promptad_evaluations

# ==============================================================================
# USER-CONTROLLABLE DATASET AND PATH SETTINGS
# ==============================================================================
DATASET_NAME = "visa"  # "mvtec", "visa", or "both"
DATASET_NAME = DATASET_NAME.lower().strip()
if DATASET_NAME not in {"mvtec", "visa", "both"}:
    raise ValueError("DATASET_NAME must be 'mvtec', 'visa', or 'both'.")
DATASETS_TO_RUN = ("mvtec", "visa") if DATASET_NAME == "both" else (DATASET_NAME,)

MVTEC_ROOT = "/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection"
VISA_ROOT = "/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922"
OUTPUT_ROOT = "/kaggle/working/outputs"
EXPECTED_CHECKPOINT_ROOT = Path(
    "/kaggle/input/promptad-few-shot-checkpoints-mvtec-ad-and-visa"
)
VERIFY_CHECKPOINT_HASHES = True

DATASET_ROOTS = {"mvtec": MVTEC_ROOT, "visa": VISA_ROOT}
DATASET_LABELS = {"mvtec": "MVTec AD", "visa": "VisA"}
for dataset_name in DATASETS_TO_RUN:
    dataset_root = DATASET_ROOTS[dataset_name]
    if not Path(dataset_root).is_dir():
        raise FileNotFoundError(
            f"{DATASET_LABELS[dataset_name]} is not mounted at {dataset_root}. "
            "Add the Kaggle dataset input or edit the corresponding path."
        )
dataset_configs = build_dataset_configs(
    mvtec_root=MVTEC_ROOT if "mvtec" in DATASETS_TO_RUN else None,
    visa_root=VISA_ROOT if "visa" in DATASETS_TO_RUN else None,
)
expected_config_names = {"MVTec" if name == "mvtec" else "VisA" for name in DATASETS_TO_RUN}
if {config.name for config in dataset_configs} != expected_config_names:
    raise RuntimeError(f"Dataset preflight did not resolve {sorted(expected_config_names)}.")
for config in dataset_configs:
    sample_count = 0
    for category in config.categories:
        probe = AnomalyDetectionDataset(config=config, category=category)
        if not probe.samples:
            raise RuntimeError(f"Dataset preflight found no test samples for {config.name}/{category}.")
        missing_masks = [
            sample["sample_id"] for sample in probe.samples
            if sample.get("is_anomaly")
            and not (sample.get("mask_path") and Path(sample["mask_path"]).is_file())
        ]
        if missing_masks:
            raise RuntimeError(
                f"Dataset preflight found {len(missing_masks)} anomalous "
                f"{config.name}/{category} samples without masks; first: {missing_masks[0]}"
            )
        sample_count += len(probe.samples)
    print(f"Dataset preflight passed: {config.name} ({sample_count} test images).")

# ==============================================================================
# USER-CONTROLLABLE CORRUPTION AND EXECUTION SETTINGS
# ==============================================================================
USE_CATEGORIZED_CORRUPTIONS = True
CORRUPTION_SEED = 123
UNCATEGORIZED_CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise",
    "defocus_blur", "motion_blur", "zoom_blur",
    "brightness", "contrast",
]
CATEGORIZED_CORRUPTION_TYPES = ["noise", "blur", "photometric", "geometric"]
CORRUPTION_TYPES = (
    CATEGORIZED_CORRUPTION_TYPES
    if USE_CATEGORIZED_CORRUPTIONS else UNCATEGORIZED_CORRUPTION_TYPES
)
# Use [] for a clean-only run. Severity 0 is reserved for clean data.
INCLUDE_CLEAN_BASELINE = True
SEVERITY_LEVELS = [1, 2, 3, 4]
SHOTS_TO_RUN = [1, 2, 4]
DEVICE = "cuda"
BATCH_SIZE = 8
CORRUPTION_CACHE_ROOT = None
CORRUPTION_CACHE_FORMAT = "png"
STRICT_SOURCE_COMMIT = True

if DEVICE != "cuda":
    raise ValueError("PromptAD's released fp16 implementation requires DEVICE='cuda'.")
if not SHOTS_TO_RUN or set(SHOTS_TO_RUN) - {1, 2, 4}:
    raise ValueError("SHOTS_TO_RUN must contain values from [1, 2, 4].")
if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be positive.")

if EXPECTED_CHECKPOINT_ROOT.is_dir():
    checkpoint_root = EXPECTED_CHECKPOINT_ROOT
else:
    index_paths = []
    for candidate in sorted(Path("/kaggle/input").rglob("checkpoint_index.json")):
        try:
            candidate_index = json.loads(candidate.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        if isinstance(candidate_index, dict) and any(
            str(key).startswith(("mvtec/", "visa/")) and "-shot/" in str(key)
            for key in candidate_index
        ):
            index_paths.append(candidate)
    if not index_paths:
        raise FileNotFoundError(
            "PromptAD checkpoint dataset is not attached. Add "
            "parsagholami/promptad-few-shot-checkpoints-mvtec-ad-and-visa."
        )
    checkpoint_root = Path(os.path.commonpath([str(path.parent) for path in index_paths]))
    print(f"Expected Kaggle slug path was absent; auto-discovered {checkpoint_root}")

CHECKPOINT_PATHS = discover_promptad_checkpoints(
    str(checkpoint_root),
    shots=SHOTS_TO_RUN,
    datasets=DATASETS_TO_RUN,
    verify_hashes=VERIFY_CHECKPOINT_HASHES,
)
checkpoint_count = sum(
    len(task_paths)
    for shot_mapping in CHECKPOINT_PATHS.values()
    for dataset_mapping in shot_mapping.values()
    for task_paths in dataset_mapping.values()
)
print(f"Checkpoint preflight passed: {checkpoint_count} indexed CLS/SEG files.")

# Fail path/checkpoint checks before downloading the ~backbone-sized LAION weights.
from PromptAD.CLIPAD.pretrained import get_pretrained_cfg, download_pretrained
backbone_cfg = get_pretrained_cfg("ViT-B-16-plus-240", "laion400m_e32")
backbone_path = download_pretrained(backbone_cfg)
print(f"Frozen PromptAD backbone ready: {backbone_path}")

print(
    f"\n====== LAUNCHING {len(SHOTS_TO_RUN)} SHOT SETTING(S) x "
    f"{len(DATASETS_TO_RUN)} DATASET(S) = "
    f"{len(SHOTS_TO_RUN) * len(DATASETS_TO_RUN)} EVALUATIONS ======"
)
print(f"Evaluation datasets: {DATASETS_TO_RUN}")
print(f"PromptAD checkpoint dataset: {checkpoint_root}")
print(f"Checkpoint hashes verified: {VERIFY_CHECKPOINT_HASHES}")
print(f"Clean baseline: {INCLUDE_CLEAN_BASELINE}")
print(f"Corruption mode: {'categorized' if USE_CATEGORIZED_CORRUPTIONS else 'uncategorized'}")
print(f"Corruptions: {CORRUPTION_TYPES}; severities: {SEVERITY_LEVELS}")
print(f"Device/batch: {DEVICE} / {BATCH_SIZE}")
print(f"Outputs: {OUTPUT_ROOT}")

run_promptad_evaluations(
    mvtec_root=MVTEC_ROOT,
    visa_root=VISA_ROOT,
    output_root=OUTPUT_ROOT,
    promptad_root=str(PROMPTAD_ROOT),
    checkpoint_paths=CHECKPOINT_PATHS,
    shots=SHOTS_TO_RUN,
    datasets=DATASETS_TO_RUN,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    corruption_types=CORRUPTION_TYPES,
    severity_levels=SEVERITY_LEVELS,
    categorized_corruptions=USE_CATEGORIZED_CORRUPTIONS,
    corruption_cache_root=CORRUPTION_CACHE_ROOT,
    corruption_cache_format=CORRUPTION_CACHE_FORMAT,
    corruption_seed=CORRUPTION_SEED,
    include_clean=INCLUDE_CLEAN_BASELINE,
    strict_source_commit=STRICT_SOURCE_COMMIT,
)

gc.collect()
torch.cuda.empty_cache()
archives = [f"PromptAD-{shot}-shot_artifacts.zip" for shot in SHOTS_TO_RUN]
print(f"Finished. Collect {len(archives)} archive(s): {archives}")
